# Tensorization and JAX

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TuLLI-AI/symprog/blob/main/notebooks/tensorization_and_jax.ipynb)

`solver.query` evaluates a circuit with a Python traversal, once per call. When the same inference is needed for **many** weight settings (e.g., weights predicted by a neural network for a batch of inputs), it is much faster to *compile* it once. This notebook covers two ways to do so:

1. **Tensorization**: compiling the symbolic expression into a batched PyTorch module.
2. **JAX** (experimental): transforming inference with `jax.jit`, `jax.vmap` and `jax.grad`.

**Note:** tensorization is **experimental**, and its API may change. `solver.tensorize` emits a `symprog.ExperimentalWarning`, which we silence in the setup below.

## Setup

We use the same formulas and weights as in `solver_tutorial.ipynb`, together with its brute-force truth table for checking results (uncomment the install line if you are running on Colab). If `symprog` is not installed, the notebook uses the version in this repository.

In [1]:
# %pip install -q "symprog[notebooks,jax]"

## use the installed symprog, or else the version in this repository
try:
    import symprog
except ImportError:
    import pathlib, sys
    repo = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "symprog" / "__init__.py").exists()), None)
    if repo is None:
        raise ImportError("symprog is not installed: pip install symprog, or run this notebook from the symprog repository")
    sys.path.insert(0, str(repo))
    import symprog

import itertools
import math
import time
import warnings

import sympy
import torch
import symprog
from symprog import Solver

## tensorization is experimental; silence its warnings for this notebook
warnings.filterwarnings("ignore", category=symprog.ExperimentalWarning)

FORMULAS = [
    "Implies(a, b)",                 # a -> b
    "And(Or(a, b), Implies(a, c))",  # (a or b) and (a -> c)
]
WEIGHTS = {"a": 0.9, "b": 0.64, "c": 0.34}

solver = Solver()
solver.add(FORMULAS)
solver.add_weights(WEIGHTS)

## brute-force truth table (see solver_tutorial.ipynb)
VARIABLES = ["a", "b", "c"]

def F(a, b, c):
    """FORMULAS as a Python function: (a -> b) and (a or b) and (a -> c)."""
    return (not a or b) and (a or b) and (not a or c)

def worlds():
    for values in itertools.product([False, True], repeat=len(VARIABLES)):
        yield dict(zip(VARIABLES, values))

def world_weight(world, w):
    return math.prod(w[v] if value else 1 - w[v] for v, value in world.items())

MODELS = [i for i in worlds() if F(**i)]

def brute_force_prob(w, evidence=None):
    """Pr(F | evidence), with evidence given as {variable: value}."""
    evidence = evidence or {}
    joint = sum(world_weight(i, w) for i in MODELS if all(i[v] == val for v, val in evidence.items()))
    return joint / math.prod(w[v] if val else 1 - w[v] for v, val in evidence.items())

# Tensorization

`solver.tensorize` computes the symbolic expression of an inference (see `symbolic_and_gradients.ipynb`) and compiles it into a PyTorch module using [`torch.fx`](https://pytorch.org/docs/stable/fx.html):

In [2]:
prob_module = solver.tensorize(mode="Prob")
prob_module.input_symbols

['a', 'b', 'c']

The module takes a batch of weight vectors, with the last dimension ordered as in `input_symbols`, and returns one result per row:

In [3]:
torch.manual_seed(0)
batch = torch.rand(5, 3)

batch_probs = prob_module(batch)
expected = [brute_force_prob(dict(zip(prob_module.input_symbols, row.tolist()))) for row in batch]
assert torch.allclose(batch_probs, torch.tensor(expected), atol=1e-6)
batch_probs

tensor([0.4207, 0.2926, 0.6573, 0.2169, 0.1662])

Inputs can also be passed per variable, by position or by name:

In [4]:
a, b, c = batch.unbind(-1)
assert torch.allclose(prob_module(a, b, c), batch_probs)
assert torch.allclose(prob_module(a=a, b=b, c=c), batch_probs)

Under the hood, the module is ordinary PyTorch code generated from the expression (shared subexpressions are computed once):

In [5]:
print(prob_module.gm.code)




def forward(self, x):
    select = torch.select(x, -1, 1)
    c0 = self.c0
    c1 = self.c1
    select_1 = torch.select(x, -1, 0)
    mul = torch.mul(c1, select_1);  c1 = None
    add = torch.add(c0, mul);  c0 = mul = None
    mul_1 = torch.mul(select, add);  add = None
    select_2 = torch.select(x, -1, 2);  x = None
    mul_2 = torch.mul(select_1, select);  select_1 = select = None
    mul_3 = torch.mul(mul_2, select_2);  mul_2 = select_2 = None
    add_1 = torch.add(mul_1, mul_3);  mul_1 = mul_3 = None
    return add_1
    


#### Other modes and evidence

Any inference mode can be tensorized. Evidence is applied when the expression is built, so the evidence variables disappear from the inputs:

In [6]:
entropy_module = solver.tensorize(mode="Entropy")
print("Entropy:", entropy_module(batch))

given_b = solver.tensorize(mode="Prob", evidence="b")
print("inputs with evidence b:", given_b.input_symbols)
expected = [brute_force_prob({"a": a_, "b": 1.0, "c": c_}, {"b": True}) for a_, c_ in zip(a.tolist(), c.tolist())]
assert torch.allclose(given_b(a=a, c=c), torch.tensor(expected), atol=1e-6)

Entropy: tensor([0.5542, 0.8968, 1.0940, 1.0748, 0.6416])
inputs with evidence b: ['a', 'c']


The options `simplify=True` and `try_horner=True` simplify the expression (the latter into [Horner form](https://en.wikipedia.org/wiki/Horner%27s_method)) before compiling. Whether this reduces the number of operations depends on the expression; for this small example, it does not:

In [7]:
def num_ops(module):
    return sum(node.op == "call_function" for node in module.gm.graph.nodes)

simplified = solver.tensorize(mode="Entropy", simplify=True, try_horner=True)
assert torch.allclose(simplified(batch), entropy_module(batch), atol=1e-5)
print("operations:", num_ops(entropy_module), "->", num_ops(simplified))

operations: 36 -> 38


#### Gradients

Tensorized modules are differentiable with PyTorch's autograd, giving gradients for a whole batch at once:

In [8]:
weights = torch.tensor([[WEIGHTS[v] for v in prob_module.input_symbols]], requires_grad=True)
(batch_grad,) = torch.autograd.grad(prob_module(weights).sum(), weights)

assert torch.allclose(batch_grad[0], solver.gradient(mode="Prob", key="weights"))
batch_grad

tensor([[-0.4224,  0.4060,  0.5760]])

#### Speed

Compare computing 1,000 probabilities one at a time, with `solver.as_function` (which traverses the circuit in Python on each call), and all at once with the tensorized module:

In [9]:
many_weights = torch.rand(1000, 3)
prob_function = solver.as_function(mode="Prob")

start = time.perf_counter()
loop_probs = [prob_function(row).item() for row in many_weights]
loop_time = time.perf_counter() - start

start = time.perf_counter()
tensor_probs = prob_module(many_weights)
tensor_time = time.perf_counter() - start

assert torch.allclose(tensor_probs, torch.tensor(loop_probs), atol=1e-5)
print(f"one at a time: {loop_time:.4f}s   tensorized: {tensor_time:.5f}s")

one at a time: 0.0628s   tensorized: 0.00007s


#### The KLay backend

With `backend="klay"`, the circuit itself (rather than its symbolic expression) is compiled with [KLay](https://github.com/ML-KULeuven/klay), which evaluates circuits layer by layer with batched tensor operations and scales to much larger circuits. It supports the `Prob`, `LogProb`, `Viterbi`, `LogViterbi` and `Fuzzy` modes, without evidence, queries or conditionals, and requires `pip install "symprog[klay]"` (Python ≥ 3.11). The resulting module is called in the same way:

In [10]:
try:
    klay_module = solver.tensorize(backend="klay")
except ImportError as e:
    klay_module = None
    print(e)

if klay_module is not None:
    print("inputs:", klay_module.input_symbols)

    start = time.perf_counter()
    klay_probs = klay_module(many_weights)
    print(f"klay: {time.perf_counter() - start:.5f}s")

    assert torch.allclose(klay_probs, tensor_probs, atol=1e-5)

    ## log probabilities, computed in log space (numerically stable for large circuits)
    klay_log_probs = solver.tensorize(mode="LogProb", backend="klay")(many_weights)
    assert torch.allclose(klay_log_probs, tensor_probs.log(), atol=1e-4)

inputs: ['a', 'b', 'c']
klay: 0.00095s


Modules can additionally be compiled with [`torch.compile`](https://pytorch.org/docs/stable/generated/torch.compile.html) via `module.compile()`, which pays off for large circuits and batches.

# JAX

[JAX](https://docs.jax.dev/en/latest/) takes a different approach: instead of compiling an expression, it transforms *functions*. The `solver.as_function` method returns inference as a function of the weight vector, and with the (**experimental**) JAX weight backend, this function can be transformed with:

- `jax.grad`: gradients,
- `jax.vmap`: vectorizing over a batch of weight vectors,
- `jax.jit`: compiling (the circuit traversal is traced once, then runs as compiled code).

This requires `pip install "symprog[jax]"`.

In [11]:
import jax
import jax.numpy as jnp

jax_solver = Solver()
jax_solver.add(FORMULAS)
jax_solver.add_weights(WEIGHTS, container_type="jax")

prob_fn = jax_solver.as_function(mode="Prob")
prob_fn(jnp.array([0.9, 0.64, 0.34]))

ModuleNotFoundError: No module named 'jax'

Composing the transformations gives batched values and batched gradients, which match the tensorized module:

In [12]:
batched_prob = jax.jit(jax.vmap(prob_fn))
batched_grad = jax.jit(jax.vmap(jax.grad(prob_fn)))

jax_batch = jnp.array(many_weights.numpy())
assert jnp.allclose(batched_prob(jax_batch), tensor_probs.numpy(), atol=1e-5)

batch_input = many_weights.clone().requires_grad_()
(torch_grads,) = torch.autograd.grad(prob_module(batch_input).sum(), batch_input)
assert jnp.allclose(batched_grad(jax_batch), torch_grads.numpy(), atol=1e-5)
batched_grad(jax_batch)[:3]

Array([[-0.13952556,  0.89630157,  0.36175585],
       [-0.0898719 ,  0.9487296 ,  0.04545356],
       [-0.04997269,  0.8848282 ,  0.36342344]], dtype=float32)

`Solver.gradient` also works with the JAX backend, and evidence and other modes are passed to `as_function` as in `query`:

In [13]:
print(jax_solver.gradient(mode="Prob", key="weights"))

entropy_given_b = jax.jit(jax.vmap(jax_solver.as_function(mode="Entropy", evidence="b")))
entropy_given_b(jax_batch[:3])

[-0.42239997  0.40600002  0.576     ]


Array([0.9591787 , 0.91151893, 0.35212404], dtype=float32)

Once compiled, the JAX functions are fast as well (the first call includes compilation time):

In [14]:
batched_prob(jax_batch).block_until_ready()  # compile

start = time.perf_counter()
batched_prob(jax_batch).block_until_ready()
print(f"jax (compiled): {time.perf_counter() - start:.5f}s")

jax (compiled): 0.00008s


**Limitations** of the experimental JAX backend: compilation time grows with the size of the circuit (the whole traversal is unrolled), and `ViterbiAssignment` cannot be used under `jax.jit` (it compares values in Python).

## Summary

| Approach | Best for |
| --- | --- |
| `solver.query` / `solver.gradient` | single inferences, exploring and debugging |
| `solver.tensorize` | batched PyTorch evaluation and gradients, e.g., inside a neural network |
| `solver.as_function` + JAX | batched evaluation and gradients with JAX transformations |